# 實驗：遷移學習

- 本章主題是遷移學習，目標是用預訓練模型加速影像分類訓練並提升準確率。
- 透過比較「ImageNet 預訓練權重」與「隨機初始化」兩種方式，驗證遷移學習通常收斂更快、表現更好。
- 資料集使用 Kaggle 的 Microsoft Cats vs Dogs Dataset https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab8.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab8.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

### 解壓縮資料集
資料壓縮檔上傳完成後，根據您實際上傳的壓縮檔名稱，替換下方指令中的 catvsdog2.zip。

In [ ]:
# 請將 "your_file.zip" 替換成你上傳的檔案名稱
!unzip catvsdog2.zip -d images_folder/

### Import必要套件

In [ ]:
!pip install torchinfo
import os
import glob
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
from torchinfo import summary

### 自訂Dataset類別

In [ ]:
class CustomDataset(Dataset):
    """自定義資料集類別，用於處理影像與標籤的載入與轉換"""
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label = self.labels[idx]
        
        # 強制轉換為 RGB 三通道，避免不同色彩空間導致的張量維度衝突
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        return image, label

### 定義獲得Dogs vs Cats資料集的Class

In [ ]:
class CatDogDataset(object):
    """處理貓狗資料集的類別，包含資料載入與 DataLoaders 產生"""
    def __init__(self, dataset_path, split_ratio=0.8, img_size=299, seed=9527, batch_size=64):
        self.img_size = (img_size, img_size)
        self.seed = seed
        self.dataset_path = dataset_path
        self.split_ratio = split_ratio
        self.batch_size = batch_size
        
        # 資料增強：加入隨機水平翻轉，提升模型泛化能力
        self.train_transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        # 驗證集不需增強，只需標準化
        self.val_transforms = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def get_dataloader(self, num_workers=0):
        # 遞迴獲取所有圖片，支援子目錄結構
        img_list = glob.glob(os.path.join(self.dataset_path, '**', '*.jpg'), recursive=True)
        
        # 標籤判斷：路徑中包含 cat 即為 1 (貓)，否則為 0 (狗)
        labels = [1 if "cat" in img_path.lower() else 0 for img_path in img_list]

        # 固定隨機種子，確保實驗可重現
        train_files, val_files, train_labels, val_labels = train_test_split(
            img_list, labels, test_size=1 - self.split_ratio, random_state=self.seed
        )

        train_dataset = CustomDataset(train_files, train_labels, transform=self.train_transforms)
        valid_dataset = CustomDataset(val_files, val_labels, transform=self.val_transforms)

        train_loader = DataLoader(train_dataset, num_workers=num_workers, batch_size=self.batch_size, shuffle=True)
        valid_loader = DataLoader(valid_dataset, num_workers=num_workers, batch_size=self.batch_size, shuffle=False)

        return train_loader, valid_loader

### 超參數與載入資料集

In [ ]:
# 偵測是否具備 GPU 加速
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"當前執行環境：{device}")

img_size = 299

dataset_path = 'images_folder/'

# 初始化資料處理實例
cat_dog_dataset = CatDogDataset(dataset_path, img_size=img_size)


train_loader, valid_loader = cat_dog_dataset.get_dataloader(num_workers=2)

### 訓練迴圈與遷移學習對比

In [ ]:
# 對比兩種模式：使用預訓練模型 (Transfer Learning) 與 不使用 (Train from Scratch)
for with_pretrained in [True, False]:
    
    # 從 Torch Hub 下載 MobileNet V3 模型架構與 ImageNet 權重
    weights = 'IMAGENET1K_V1' if with_pretrained else None
    model = torch.hub.load('pytorch/vision', 'mobilenet_v3_large', weights=weights)

    # 如果是遷移學習模式，凍結特徵擷取層 (卷積層) 的權重，不參與更新
    if with_pretrained:
        for param in model.parameters():
            param.requires_grad = False

    # 關鍵步驟：替換最後一層分類器為 1 個輸出節點 (二元分類)
    num_features = model.classifier[3].in_features
    model.classifier[3] = torch.nn.Linear(num_features, 1)
    model = model.to(device)

    # 定義 Loss 函數 (BCEWithLogitsLoss 已內建 Sigmoid)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    # 若使用預訓練模型，優化器僅需更新最後一層分類器的參數
    params = model.classifier[3].parameters() if with_pretrained else model.parameters()
    optimizer = torch.optim.AdamW(params, lr=0.001)

    print(f'\n開始執行：{"[遷移學習模式]" if with_pretrained else "[從頭訓練模式]"}')
    num_epochs = 1 if with_pretrained else 5
    
    for epoch in range(num_epochs):
        # --- 訓練階段 ---
        model.train()
        tqdm_train = tqdm(train_loader, desc=f'Epoch {epoch+1} Train', dynamic_ncols=True)
        for images, labels in tqdm_train:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs.squeeze(), labels.float())
            loss.backward()
            optimizer.step()
            tqdm_train.set_postfix(loss=loss.item())

        # --- 驗證階段 ---
        model.eval()
        accuracy = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                # 預測值 > 0 即代表正類 (貓)
                preds = (outputs.squeeze() > 0)
                accuracy += torch.sum(preds == labels.byte()).item()

        acc_percent = accuracy / len(valid_loader.dataset) * 100
        print(f'Epoch {epoch+1} 驗證準確率: {acc_percent:.2f}%')